# LightGBM 5-Fold Cross-Validation for Rome LST

This notebook is fully self-contained. It:

1. loads `rome_dataset_features_target_with_focal.csv`;
2. excludes `LST` and the coordinate columns from the predictors;
3. runs shuffled 5-fold cross-validation with the provided LightGBM hyperparameters;
4. calculates R², MAE and MAPE for train and validation in every fold;
5. saves fold metrics, summary statistics and feature importances as CSV files.

The split is random. Therefore, these results measure random pixel interpolation within Rome, not generalisation to geographically unseen areas.

In [ ]:
from pathlib import Path
from time import perf_counter
import importlib.util

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import KFold

if importlib.util.find_spec("lightgbm") is None:
    raise ImportError("LightGBM is not installed. Run: %pip install lightgbm")

from lightgbm import LGBMRegressor

RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "LST"
COORDINATE_COLUMNS = ["x_utm", "y_utm", "lon", "lat"]
OUTPUT_DIR = Path("model_outputs/lightgbm_5fold_cv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PARAMS = {
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": -1,
    "n_estimators": 300,
    "num_leaves": 24,
    "min_child_samples": 128,
    "reg_lambda": 0.03857720932178909,
    "reg_alpha": 0.21598801171228973,
    "subsample": 0.8347063033889262,
    "colsample_bytree": 0.8759561732083417,
    "learning_rate": 0.01879471459020519,
}

print("Output directory:", OUTPUT_DIR.resolve())
print("LightGBM parameters:", BEST_PARAMS)

In [ ]:
def find_dataset():
    candidates = [
        Path("/Users/simon/Downloads/rome_dataset_features_target_with_focal.csv"),
        Path("../../rome_dataset_features_target_with_focal.csv"),
        Path("../rome_dataset_features_target_with_focal.csv"),
        Path("rome_dataset_features_target_with_focal.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "rome_dataset_features_target_with_focal.csv was not found. "
        "Place it next to the notebook or update the candidate paths."
    )


def infer_features(dataframe):
    excluded = {TARGET, *COORDINATE_COLUMNS}
    features = [
        column
        for column in dataframe.select_dtypes(include=np.number).columns
        if column not in excluded
    ]
    if not features:
        raise ValueError("No numeric predictor columns were found.")
    return features


def calculate_metrics(y_true, y_pred):
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mape),
        "MAPE_percent": float(100 * mape),
    }


DATA_PATH = find_dataset()
print("Dataset:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
FEATURES = infer_features(df)

required_columns = FEATURES + [TARGET]
missing_rows = int(df[required_columns].isna().any(axis=1).sum())
if missing_rows:
    print(f"Removing {missing_rows:,} rows containing missing model values.")
    df = df.dropna(subset=required_columns).reset_index(drop=True)

X = df[FEATURES]
y = df[TARGET]

print(f"Rows: {len(df):,}")
print(f"Features: {len(FEATURES)}")
print("Coordinates excluded from predictors:", COORDINATE_COLUMNS)
display(df[FEATURES + [TARGET]].describe().T)

In [ ]:
def run_lightgbm_cross_validation(X, y, features, params, n_splits=5, random_state=42):
    splitter = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    metric_rows = []
    importance_rows = []
    models = []

    for fold_number, (train_idx, validation_idx) in enumerate(splitter.split(X), start=1):
        started = perf_counter()
        model = LGBMRegressor(**params)
        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
        )
        elapsed_seconds = perf_counter() - started

        train_prediction = model.predict(X.iloc[train_idx])
        validation_prediction = model.predict(X.iloc[validation_idx])

        for subset, indices, prediction in (
            ("train", train_idx, train_prediction),
            ("validation", validation_idx, validation_prediction),
        ):
            metric_rows.append({
                "fold": fold_number,
                "subset": subset,
                "rows": len(indices),
                "n_features": len(features),
                "elapsed_seconds": elapsed_seconds,
                **calculate_metrics(y.iloc[indices], prediction),
            })

        for feature, importance in zip(features, model.feature_importances_):
            importance_rows.append({
                "fold": fold_number,
                "feature": feature,
                "importance": float(importance),
            })

        models.append(model)
        validation_metrics = metric_rows[-1]
        print(
            f"Fold {fold_number}/{n_splits} | "
            f"R2={validation_metrics['R2']:.4f} | "
            f"MAE={validation_metrics['MAE']:.4f} C | "
            f"MAPE={validation_metrics['MAPE_percent']:.4f}% | "
            f"time={elapsed_seconds:.2f}s"
        )

    return pd.DataFrame(metric_rows), pd.DataFrame(importance_rows), models


metrics_df, feature_importance_df, models = run_lightgbm_cross_validation(
    X,
    y,
    FEATURES,
    BEST_PARAMS,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

In [ ]:
metrics_summary_df = (
    metrics_df.groupby("subset")[["R2", "MAE", "MAPE", "MAPE_percent", "elapsed_seconds"]]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
metrics_summary_df.columns = [
    "_".join(str(part) for part in column if part)
    if isinstance(column, tuple)
    else column
    for column in metrics_summary_df.columns
]

feature_importance_summary_df = (
    feature_importance_df.assign(
        rank=feature_importance_df.groupby("fold")["importance"].rank(
            ascending=False,
            method="average",
        )
    )
    .groupby("feature")
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
        importance_min=("importance", "min"),
        importance_max=("importance", "max"),
        mean_rank=("rank", "mean"),
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index()
)

METRICS_PATH = OUTPUT_DIR / "lightgbm_5fold_metrics.csv"
SUMMARY_PATH = OUTPUT_DIR / "lightgbm_5fold_metrics_summary.csv"
IMPORTANCE_PATH = OUTPUT_DIR / "lightgbm_5fold_feature_importance.csv"
IMPORTANCE_SUMMARY_PATH = OUTPUT_DIR / "lightgbm_5fold_feature_importance_summary.csv"
PARAMETERS_PATH = OUTPUT_DIR / "lightgbm_parameters.csv"

metrics_df.to_csv(METRICS_PATH, index=False)
metrics_summary_df.to_csv(SUMMARY_PATH, index=False)
feature_importance_df.to_csv(IMPORTANCE_PATH, index=False)
feature_importance_summary_df.to_csv(IMPORTANCE_SUMMARY_PATH, index=False)
pd.DataFrame([BEST_PARAMS]).to_csv(PARAMETERS_PATH, index=False)

print("\nCSV files saved successfully:")
for output_path in (
    METRICS_PATH,
    SUMMARY_PATH,
    IMPORTANCE_PATH,
    IMPORTANCE_SUMMARY_PATH,
    PARAMETERS_PATH,
):
    print(" -", output_path.resolve())

print("\nValidation metrics by fold:")
display(metrics_df.query("subset == 'validation'")[["fold", "rows", "R2", "MAE", "MAPE_percent"]])

print("\nCross-validation summary:")
display(metrics_summary_df)

print("\nTop 20 LightGBM key drivers:")
display(feature_importance_summary_df.head(20))

## Output files

After execution, the notebook creates:

- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_metrics.csv`: train and validation metrics for every fold;
- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_metrics_summary.csv`: mean, standard deviation, minimum and maximum metrics;
- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_feature_importance.csv`: feature importance for every fold;
- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_feature_importance_summary.csv`: stable key-driver ranking;
- `model_outputs/lightgbm_5fold_cv/lightgbm_parameters.csv`: exact model parameters used.